In [1]:
!pip install xarray cfgrib netCDF4 pandas requests zarr dask

In [2]:
import os
import requests
import xarray as xr
import pandas as pd
from pathlib import Path
from urllib.parse import urlencode

In [3]:
# ==============================
# USER CONFIGURATION
# ==============================

FEATURES = [
    "temperature",
    "pressure"
]

PARAMETERS = {
    "temperature": "2 m above ground",
    "pressure": "Mean Sea Level Pressure"
}

LATITUDE = (5, 35)
LONGITUDE = (65, 100)

# GFS settings
DATE = "20260910"
CYCLE = "00"
FORECAST_HOUR = "000"

# Output
OUTPUT_DIR = Path("gfs_output")

OUTPUT_DIR.mkdir(exist_ok=True)

In [4]:
# ==============================
# GFS FEATURE MAPPING
# ==============================

FEATURE_MAP = {
    "temperature": {
        "variable": "TMP",
        "level": "lev_2_m_above_ground"
    },

    "pressure": {
        "variable": "PRMSL",
        "level": "lev_mean_sea_level"
    }
}

In [5]:
# ==============================
# BUILD GFS REQUEST URL
# ==============================

def build_gfs_url():
    
    variables = set()
    levels = set()

    for feature in FEATURES:
        
        if feature not in FEATURE_MAP:
            raise ValueError(f"Unsupported feature: {feature}")
        
        variables.add(
            FEATURE_MAP[feature]["variable"]
        )
        
        levels.add(
            FEATURE_MAP[feature]["level"]
        )

    params = {
        "file": f"gfs.t{CYCLE}z.pgrb2.0p25.f{FORECAST_HOUR}",

        "all_lev": "off",
        "all_var": "off",

        "subregion": "",

        "leftlon": LONGITUDE[0],
        "rightlon": LONGITUDE[1],

        "toplat": LATITUDE[1],
        "bottomlat": LATITUDE[0],

        "dir": f"/gfs.{DATE}/{CYCLE}/atmos"
    }

    # Add variables
    for var in variables:
        params[f"var_{var}"] = "on"

    # Add levels
    for level in levels:
        params[level] = "on"

    base_url = "https://nomads.ncep.noaa.gov/cgi-bin/filter_gfs_0p25.pl"

    return base_url + "?" + urlencode(params)

In [6]:
url = build_gfs_url()

print(url)

https://nomads.ncep.noaa.gov/cgi-bin/filter_gfs_0p25.pl?file=gfs.t00z.pgrb2.0p25.f000&all_lev=off&all_var=off&subregion=&leftlon=65&rightlon=100&toplat=35&bottomlat=5&dir=%2Fgfs.20260910%2F00%2Fatmos&var_TMP=on&var_PRMSL=on&lev_mean_sea_level=on&lev_2_m_above_ground=on


In [7]:
# ==============================
# DOWNLOAD GFS DATA
# ==============================

def download_gfs(url):
    
    output_file = OUTPUT_DIR / "gfs_data.grib2"

    print("Downloading GFS data...")
    print(url)

    response = requests.get(url, timeout=120)

    response.raise_for_status()

    with open(output_file, "wb") as f:
        f.write(response.content)

    print("Download completed.")
    print(f"Saved at: {output_file}")

    return output_file

In [8]:
grib_file = download_gfs(url)

https://nomads.ncep.noaa.gov/cgi-bin/filter_gfs_0p25.pl?file=gfs.t00z.pgrb2.0p25.f000&all_lev=off&all_var=off&subregion=&leftlon=65&rightlon=100&toplat=35&bottomlat=5&dir=%2Fgfs.20260910%2F00%2Fatmos&var_TMP=on&var_PRMSL=on&lev_mean_sea_level=on&lev_2_m_above_ground=on
Download completed.
Saved at: gfs_output\gfs_data.grib2


In [9]:
print(xr.__version__)

2026.7.0


In [10]:
temperature = xr.open_dataset(
    grib_file,
    engine="cfgrib",
    backend_kwargs={
        "filter_by_keys": {
            "shortName": "2t"
        }
    }
)

print("Temperature dataset successfully opened!")
print(temperature.data_vars)
print(temperature.coords)

Ignoring index file 'gfs_output\\gfs_data.grib2.47d85.idx' older than GRIB file


Temperature dataset successfully opened!
Data variables:
    t2m      (latitude, longitude) float32 68kB ...
Coordinates:
  * latitude           (latitude) float64 968B 5.0 5.25 5.5 ... 34.5 34.75 35.0
  * longitude          (longitude) float64 1kB 65.0 65.25 65.5 ... 99.75 100.0
    time               datetime64[ns] 8B ...
    step               timedelta64[ns] 8B ...
    heightAboveGround  float64 8B ...
    valid_time         datetime64[ns] 8B ...


In [11]:
pressure = xr.open_dataset(
    grib_file,
    engine="cfgrib",
    backend_kwargs={
        "filter_by_keys": {
            "shortName": "prmsl"
        }
    }
)

print("Pressure dataset successfully opened!")
print(pressure.data_vars)
print(pressure.coords)

Ignoring index file 'gfs_output\\gfs_data.grib2.47d85.idx' older than GRIB file


Pressure dataset successfully opened!
Data variables:
    prmsl    (latitude, longitude) float32 68kB ...
Coordinates:
  * latitude    (latitude) float64 968B 5.0 5.25 5.5 5.75 ... 34.5 34.75 35.0
  * longitude   (longitude) float64 1kB 65.0 65.25 65.5 ... 99.5 99.75 100.0
    time        datetime64[ns] 8B ...
    step        timedelta64[ns] 8B ...
    meanSea     float64 8B ...
    valid_time  datetime64[ns] 8B ...


In [12]:
# Temperature: Kelvin → Celsius
temperature_c = temperature["t2m"] - 273.15

# Pressure: Pa → hPa
pressure_hpa = pressure["prmsl"] / 100

print("Temperature range:")
print(float(temperature_c.min()), "to", float(temperature_c.max()), "°C")

print("\nPressure range:")
print(float(pressure_hpa.min()), "to", float(pressure_hpa.max()), "hPa")

Temperature range:
-8.4744873046875 to 31.925506591796875 °C

Pressure range:
1002.6375122070312 to 1029.3695068359375 hPa


In [13]:
final_dataset = xr.Dataset(
    {
        "temperature_2m_C": temperature_c,
        "pressure_msl_hPa": pressure_hpa
    }
)

print("Final dataset created!")
print(final_dataset)

Final dataset created!
<xarray.Dataset> Size: 139kB
Dimensions:            (latitude: 121, longitude: 141)
Coordinates:
  * latitude           (latitude) float64 968B 5.0 5.25 5.5 ... 34.5 34.75 35.0
  * longitude          (longitude) float64 1kB 65.0 65.25 65.5 ... 99.75 100.0
    time               datetime64[ns] 8B 2026-09-10
    step               timedelta64[ns] 8B 00:00:00
    heightAboveGround  float64 8B ...
    valid_time         datetime64[ns] 8B 2026-09-10
    meanSea            float64 8B ...
Data variables:
    temperature_2m_C   (latitude, longitude) float32 68kB 25.33 25.63 ... 3.026
    pressure_msl_hPa   (latitude, longitude) float32 68kB 1.012e+03 ... 1.026...


In [14]:
# ==============================
# ZARR OUTPUT
# ==============================

zarr_store = OUTPUT_DIR / "gfs_weather_data.zarr"

final_dataset.to_zarr(
    zarr_store,
    mode="w"
)

print("Zarr dataset saved successfully!")
print("Path:", zarr_store)

Zarr dataset saved successfully!
Path: gfs_output\gfs_weather_data.zarr


C:\Users\adars\anaconda3\Lib\site-packages\zarr\api\asynchronous.py:246: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


In [15]:
# ==============================
# ZARR VALIDATION / PREVIEW
# ==============================

ds = xr.open_zarr(zarr_store)

print("Zarr dataset loaded successfully!")
print(ds)

print("\nDimensions:")
print(ds.dims)

print("\nCoordinates:")
print(ds.coords)

print("\nVariables:")
print(ds.data_vars)

print("\nTemperature:")
print(ds["temperature_2m_C"])

print("\nPressure:")
print(ds["pressure_msl_hPa"])

Zarr dataset loaded successfully!
<xarray.Dataset> Size: 139kB
Dimensions:            (latitude: 121, longitude: 141)
Coordinates:
  * latitude           (latitude) float64 968B 5.0 5.25 5.5 ... 34.5 34.75 35.0
  * longitude          (longitude) float64 1kB 65.0 65.25 65.5 ... 99.75 100.0
    heightAboveGround  float64 8B ...
    meanSea            float64 8B ...
    step               timedelta64[ns] 8B ...
    time               datetime64[ns] 8B ...
    valid_time         datetime64[ns] 8B ...
Data variables:
    pressure_msl_hPa   (latitude, longitude) float32 68kB dask.array<chunksize=(121, 141), meta=np.ndarray>
    temperature_2m_C   (latitude, longitude) float32 68kB dask.array<chunksize=(121, 141), meta=np.ndarray>

Dimensions:
FrozenMappingWarningOnValuesAccess({'latitude': 121, 'longitude': 141})

Coordinates:
Coordinates:
  * latitude           (latitude) float64 968B 5.0 5.25 5.5 ... 34.5 34.75 35.0
  * longitude          (longitude) float64 1kB 65.0 65.25 65.5 ... 99.75 1

In [21]:
zarr_store = OUTPUT_DIR / "gfs_weather_data.zarr"

ds = xr.open_zarr(zarr_store)

print(ds)

<xarray.Dataset> Size: 139kB
Dimensions:            (latitude: 121, longitude: 141)
Coordinates:
  * latitude           (latitude) float64 968B 5.0 5.25 5.5 ... 34.5 34.75 35.0
  * longitude          (longitude) float64 1kB 65.0 65.25 65.5 ... 99.75 100.0
    heightAboveGround  float64 8B ...
    meanSea            float64 8B ...
    step               timedelta64[ns] 8B ...
    time               datetime64[ns] 8B ...
    valid_time         datetime64[ns] 8B ...
Data variables:
    pressure_msl_hPa   (latitude, longitude) float32 68kB dask.array<chunksize=(121, 141), meta=np.ndarray>
    temperature_2m_C   (latitude, longitude) float32 68kB dask.array<chunksize=(121, 141), meta=np.ndarray>


In [22]:
print(ds["temperature_2m_C"])
print(ds["pressure_msl_hPa"])

<xarray.DataArray 'temperature_2m_C' (latitude: 121, longitude: 141)> Size: 68kB
dask.array<open_dataset-temperature_2m_C, shape=(121, 141), dtype=float32, chunksize=(121, 141), chunktype=numpy.ndarray>
Coordinates:
  * latitude           (latitude) float64 968B 5.0 5.25 5.5 ... 34.5 34.75 35.0
  * longitude          (longitude) float64 1kB 65.0 65.25 65.5 ... 99.75 100.0
    heightAboveGround  float64 8B ...
    meanSea            float64 8B ...
    step               timedelta64[ns] 8B ...
    time               datetime64[ns] 8B ...
    valid_time         datetime64[ns] 8B ...
Attributes: (12/30)
    GRIB_paramId:                             167
    GRIB_dataType:                            fc
    GRIB_numberOfPoints:                      17061
    GRIB_typeOfLevel:                         heightAboveGround
    GRIB_stepUnits:                           1
    GRIB_stepType:                            instant
    ...                                       ...
    GRIB_name:            